# Book-to-Movie Recommender · Colab 실행
런타임 → 런타임 유형 변경에서 NVIDIA GPU를 선택한 뒤 위에서 아래로 셀을 실행한다. 이 노트북은 CF 20%, 추론 책 최대 50권 조건에서 네 모델을 평가한다.
데이터와 모델은 본인의 Google Drive에 준비한다. 실행 중에는 Colab 로컬 디스크를 사용하고, 완료된 단계만 Drive에 복사한다.

## 1. GPU와 런타임 확인
GPU가 나오지 않으면 런타임 유형을 확인한다. 드라이버 정보와 CUDA 도구 버전을 구분해 확인한다.

In [6]:
import shutil
import subprocess
import sys
from pathlib import Path

print("Python", sys.version)
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Python 3.10~3.13 Colab 런타임을 선택하세요. 예: 2026.07")
subprocess.run(["nvidia-smi"], check=True)
if shutil.which("nvcc"):
    subprocess.run(["nvcc", "--version"], check=True)
subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

Python 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


CompletedProcess(args=['df', '-h', '/content'], returncode=0)

## 2. 코드 가져오기와 패키지 설치
별도 가상환경에 GPU용 ONNX Runtime과 CUDA·cuDNN 라이브러리를 설치한다. 이 환경에는 CPU용 ONNX Runtime 패키지를 함께 설치하지 않는다.

In [ ]:
import json

REPO = Path("/content/book-to-movie-recommender")
REPO_URL = "https://github.com/sldfkajsfd/book-to-movie-recommender.git"
BRANCH = "colab-cuda"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
else:
    current = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO, text=True).strip()
    if current != BRANCH:
        raise RuntimeError(f"기존 {REPO}가 {BRANCH} 브랜치에 있지 않습니다.")
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, check=True)

VENV = Path("/content/book-movie-venv")
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV)], check=True)
from collections import deque

install_log = Path("/content/book_movie_install.log")
last_lines = deque(maxlen=25)
# The runtime pip can target a virtual environment before it has its own pip.
install = [sys.executable, "-m", "pip", "--python", str(PYTHON),
           "install", "--no-cache-dir", "pip", "-r", str(REPO / "requirements-colab.txt")]
with install_log.open("w", encoding="utf-8") as log:
    with subprocess.Popen(install, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            log.write(line)
            last_lines.append(line)
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError(f"설치 실패. 로그 {install_log}\n" + "".join(last_lines))
packages = json.loads(subprocess.check_output(
    [str(PYTHON), "-m", "pip", "list", "--format=json"], text=True))
for package in packages:
    if package["name"].startswith(("nvidia-", "onnxruntime")):
        print(package["name"], package["version"])

## 3. Drive 연결과 입력 준비
Drive 연결을 직접 승인한다. 아래 폴더명이 다르면 `DRIVE_ROOT`를 바꾼다. `data/`에 입력 5개, `models/minilm/`에 `tokenizer.json`과 FP32 `model.onnx`를 둔다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/book-to-movie-recommender")
RUN_NAME = "ablation_200k_cf20_top50_cuda"
BATCH_SIZE = 32
DRIVE_OUTPUT = DRIVE_ROOT / "runs" / RUN_NAME
LOCAL_DATA = Path("/content/book_movie_data")
LOCAL_MODELS = Path("/content/book_movie_minilm")
LOCAL_OUTPUT = Path("/content/book_movie_runs") / RUN_NAME

input_files = [
    "books_ratings.csv", "movies_ratings.csv", "books_tfidf_metadata.jsonl",
    "movies_tfidf_metadata.jsonl", "users_content_filtered.csv",
]
inputs = [(DRIVE_ROOT / "data" / name, LOCAL_DATA / name) for name in input_files]
inputs += [(DRIVE_ROOT / "models/minilm" / name, LOCAL_MODELS / name)
           for name in ("tokenizer.json", "model.onnx")]
missing = [str(source) for source, _ in inputs if not source.is_file() or source.stat().st_size == 0]
if missing:
    raise FileNotFoundError("\n".join(missing))
for source, target in inputs:
    target.parent.mkdir(parents=True, exist_ok=True)
    # A failed copy never replaces a complete local file.
    partial = target.with_suffix(target.suffix + ".partial")
    shutil.copy2(source, partial)
    partial.replace(target)
    print(target.name, f"{target.stat().st_size / 2**20:.1f} MiB")

def restore(local, saved):
    if not local.exists() and saved.is_dir():
        import tempfile
        local.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=local.parent) as tmp:
            copy = Path(tmp) / "checkpoint"
            shutil.copytree(saved, copy, ignore=shutil.ignore_patterns("*.partial"))
            copy.replace(local)
        print("복구", local)
    local.mkdir(parents=True, exist_ok=True)

restore(LOCAL_OUTPUT, DRIVE_OUTPUT)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
environment = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    "python": subprocess.check_output([str(PYTHON), "-V"], text=True).strip(),
    "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True),
    "packages": json.loads(subprocess.check_output(
        [str(PYTHON), "-m", "pip", "list", "--format=json"], text=True)),
}
(DRIVE_OUTPUT / "runtime_environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

## 4. 실제 CUDA 연산과 결과 비교
동일한 FP32 ONNX 파일에 같은 표본을 넣어 CPU와 CUDA를 비교한다. CUDA 연산 이벤트, 최소 코사인 유사도 0.999, 예시 프로필당 Top 10의 최소 9개 공통 추천을 점검한다. 목록이 더 짧으면 두 경로가 같은 수를 반환하고 모두 겹치는지 확인한다. 검증에 실패하면 전체 실행으로 넘어가기 전에 원인을 확인한다.

In [ ]:
def run_support(action, local=LOCAL_OUTPUT, saved=DRIVE_OUTPUT, device="cuda"):
    command = [str(PYTHON), "colab_support.py", action,
               "--data", str(LOCAL_DATA), "--model-dir", str(LOCAL_MODELS),
               "--output", str(local), "--drive-output", str(saved),
               "--batch-size", str(BATCH_SIZE), "--device", device]
    from collections import deque

    local.mkdir(parents=True, exist_ok=True)
    log_path = local / f"{action}.log"
    last_lines = deque(maxlen=35)
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, encoding="utf-8", errors="replace") as process:
            for line in process.stdout:
                log.write(line)
                log.flush()
                last_lines.append(line)
                print(line, end="", flush=True)
            if process.wait():
                raise RuntimeError(f"{action} 실패. 실행 로그 {log_path}\n" + "".join(last_lines))

run_support("check")

## 5. 네 모델 실행
MiniLM ONNX 추론에만 NVIDIA GPU를 사용한다. TF-IDF, 사용자 KNN, 평균 풀링과 정규화, 추천 순위와 평가는 CPU에서 수행한다. GPU 메모리가 부족하면 배치 크기를 줄이고 새 실행 이름을 사용한다.

In [ ]:
if not json.loads((LOCAL_OUTPUT / "cuda_check.json").read_text(encoding="utf-8"))["passed"]:
    raise RuntimeError("CUDA 비교 셀을 먼저 통과해야 합니다.")
run_support("run")

## 6. 평가와 자원 사용량 확인
추천 CSV와 지표, 자원 표본, 완료된 캐시는 실행 중 Drive에 복사된다. 세션이 끝나면 같은 실행 이름으로 3단계부터 복구한다. 작업 중이던 단계는 다시 계산한다.

In [ ]:
for name in ("metrics.json", "ranking_checks.json", "run_stats.json", "resource_stats.json"):
    print(f"\n{name}")
    print((LOCAL_OUTPUT / name).read_text(encoding="utf-8"))
print("\nDrive 저장 위치", DRIVE_OUTPUT)
for file in sorted(DRIVE_OUTPUT.glob("*_recommendations.csv")):
    print(file.name, f"{file.stat().st_size / 2**20:.1f} MiB")

## 7. 선택 · 전체 FP32 CPU 비교
전체 데이터의 가속 전후를 측정할 때만 아래의 `RUN_CPU_BASELINE`을 켠다. CUDA와 같은 FP32 모델을 사용하며, 공통 TF-IDF와 책 CF 캐시를 재사용한다. 최초 실행 전체 시간과 별도로, 캐시 여부가 같은 단계끼리 시간을 비교한다. CPU 전체 임베딩에는 시간이 오래 걸릴 수 있다.

In [ ]:
RUN_CPU_BASELINE = False
if RUN_CPU_BASELINE:
    cpu_name = RUN_NAME + "_cpu_fp32"
    cpu_output = LOCAL_OUTPUT.with_name(cpu_name)
    cpu_drive = DRIVE_OUTPUT.with_name(cpu_name)
    restore(cpu_output, cpu_drive)
    cpu_cache = cpu_output / "cache"
    cpu_cache.mkdir(exist_ok=True)
    shared = ["inputs.json", "books_tfidf.npz", "movies_tfidf.npz", "catalog.npz",
              "tfidf.joblib", "tfidf_complete.json", "inferred_books_50.npz"]
    for name in shared:
        if not (cpu_cache / name).exists():
            shutil.copy2(LOCAL_OUTPUT / "cache" / name, cpu_cache / name)
    run_support("run", cpu_output, cpu_drive, "cpu_fp32")
    for output in (LOCAL_OUTPUT, cpu_output):
        print(output.name)
        print((output / "run_stats.json").read_text(encoding="utf-8"))
        print((output / "resource_stats.json").read_text(encoding="utf-8"))